In [ ]:
import os
import glob
import pandas as pd
import anndata as ad
import scirpy as ir

DATA_DIR = str(DATA_ROOT / "GSE295491")
BCR_OUT  = f"{DATA_DIR}/combined_bcr.h5ad"


In [ ]:

# BCR sample metadata
bcr_titles = [
    "P19 HC-MBL BCR","P19 CLL BCR","P21 HC-MBL BCR","P21 CLL BCR",
    "P13 HC-MBL BCR","P11 HC-MBL BCR","P22 HC-MBL BCR","P22 CLL BCR",
    "P15 HC-MBL BCR","P15 CLL BCR","P14 HC-MBL BCR","P14 CLL BCR",
    "P8 HC-MBL BCR","P10 HC-MBL BCR","P7 HC-MBL BCR","P9 HC-MBL BCR",
    "P23 HC-MBL BCR","P23 CLL BCR","P20 HC-MBL BCR","P20 CLL BCR",
    "P16 HC-MBL BCR","P16 CLL BCR","P12 HC-MBL BCR","P24 CLL BCR",
    "P6 LC-MBL BCR","P9 HC-MBL BCR replicate","P2 LC-MBL BCR",
    "P1 LC-MBL BCR","P4 LC-MBL BCR","P3 LC-MBL BCR","P5 LC-MBL BCR"
]

bcr_gsms = [
    "GSM8951259","GSM8951260","GSM8951261","GSM8951262","GSM8951263","GSM8951264",
    "GSM8951265","GSM8951266","GSM8951267","GSM8951268","GSM8951269","GSM8951270",
    "GSM8951271","GSM8951272","GSM8951273","GSM8951274","GSM8951275","GSM8951276",
    "GSM8951277","GSM8951278","GSM8951279","GSM8951280","GSM8951281","GSM8951282",
    "GSM8951283","GSM8951284","GSM8951285","GSM8951286","GSM8951287","GSM8951288",
    "GSM8951289"
]

bcr_meta = pd.DataFrame({"gsm": bcr_gsms, "title": bcr_titles})
bcr_meta["patient_id"]    = bcr_meta["title"].str.extract(r"(P\d+)")
bcr_meta["disease_stage"] = bcr_meta["title"].str.extract(r"(HC-MBL|LC-MBL|CLL)")
bcr_meta["is_replicate"]  = bcr_meta["title"].str.contains("replicate")

#  skip replicate first pass
bcr_meta = bcr_meta.loc[~bcr_meta["is_replicate"]].copy()

bcr_adatas = []
keys = []

for _, row in bcr_meta.iterrows():
    gsm = row["gsm"]
    title = row["title"]

    candidates = glob.glob(f"{DATA_DIR}/{gsm}*filtered_contig_annotations*.csv*")
    if len(candidates) == 0:
        print(f"MISSING BCR FILE: {gsm} {title}")
        continue

    path = candidates[0]
    print(f"Reading {gsm}: {os.path.basename(path)}")

    adata_bcr = ir.io.read_10x_vdj(path)

    # prefix obs_names to match RNA object convention in other script
    adata_bcr.obs_names = [f"{gsm}_{bc}" for bc in adata_bcr.obs_names]

    # add metadata
    adata_bcr.obs["gsm"] = gsm
    adata_bcr.obs["patient_id"] = row["patient_id"]
    adata_bcr.obs["disease_stage"] = row["disease_stage"]
    adata_bcr.obs["sample_title"] = title

    bcr_adatas.append(adata_bcr)
    keys.append(gsm)

print(f"Loaded {len(bcr_adatas)} BCR samples")

adata_bcr = ad.concat(
    bcr_adatas,
    axis=0,
    join="outer",
    merge="same",
    label="gsm_concat",
    keys=keys,
    index_unique=None
)

print(adata_bcr)
adata_bcr.write_h5ad(BCR_OUT)
print(f"Saved {BCR_OUT}")

In [ ]:
adata_bcr.obs

In [ ]:
adata_bcr.obsm['airr']

In [ ]:
print(adata_bcr.obsm['airr']['locus'].value_counts())

In [ ]:
import scanpy as sc
adata_bcr = sc.read_h5ad(BCR_OUT)

In [ ]:
ir.pp.index_chains(adata_bcr, filter=["productive", "require_junction_aa"])

In [ ]:
ir.tl.chain_qc(adata_bcr)

In [ ]:
ir.pp.ir_dist(
    adata_bcr,
    metric="normalized_hamming",
    cutoff=15,            # 85% junction similarity threshold
    sequence="nt",        # nucleotide (not aa) — SHM acts at nt level
    histogram=True        # shows bimodal dist to validate cutoff
)


In [ ]:
ir.tl.define_clonotype_clusters(
    adata_bcr,
    inplace=True,
    metric="normalized_hamming",
    sequence="nt",
    receptor_arms="all",
    same_v_gene=True,
    same_j_gene=True,
    dual_ir="any",
    within_group="patient_id",
    key_added="clone_id",          # ensure consistent name
)

In [ ]:
print("obs columns:", list(adata_bcr.obs.columns))
print(adata_bcr.obs.filter(like="clone"))

In [ ]:
# adjust this if your printout shows a slightly different name
clone_col = "clone_id"  # or "clone_id" depending on what you saw above

ir.tl.clonal_expansion(
    adata_bcr,
    target_col=clone_col,
    breakpoints=(1, 2),
)

print(adata_bcr.obs.filter(like="clonal_expansion").head())
print(adata_bcr.obs["clonal_expansion"].value_counts())

In [ ]:
clone_col = "clone_id"

In [ ]:
ir.pl.clonal_expansion(
    adata_bcr,
    target_col=clone_col,     # airr:clone_id or clone_id
    breakpoints=(1, 2),
    groupby="disease_stage",
    normalize=True,
)

In [ ]:
ir.pl.group_abundance(
    adata_bcr,
    groupby="disease_stage",
    target_col="clonal_expansion",
    normalize=True,
)

In [ ]:
# ensure consistent order of expansion categories and stages
ir.pl.clonal_expansion(
    adata_bcr,
    target_col="clone_id",          # or airr:clone_id if that's the name
    breakpoints=(1, 2),
    groupby="disease_stage",
    normalize=True,
)

In [ ]:

airr = adata_bcr.obsm["airr"]
print(type(airr))
print(airr.fields)          # list of available AIRR fields (like v_call, j_call, etc.)
print(airr["v_call"][:5])

In [ ]:
with ir.get.airr_context(adata_bcr, "v_call"):
    print("Obs columns now containing v_call:")
    print([c for c in adata_bcr.obs.columns if "v_call" in c][:10])
    print(adata_bcr.obs.filter(like="v_call").head())

In [ ]:
# replace VDJ_1_v_call with whatever you actually saw above
with ir.get.airr_context(adata_bcr, "v_call"):
    ir.pl.group_abundance(
        adata_bcr,
        groupby="disease_stage",   # HC-MBL / LC-MBL / CLL
        target_col="VDJ_1_v_call",
        normalize=True,
        max_cols=20,
    )

In [ ]:

cll_genes = ["IGHV1-69", "IGHV4-34", "IGHV3-30", "IGHV3-21", "IGHV1-2"]

# Step 1: extract heavy-chain V gene as a plain Series
with ir.get.airr_context(adata_bcr, "v_call"):
    v_heavy = adata_bcr.obs["VDJ_1_v_call"].astype("string").copy()

# Step 2: assign IGHV_CLL_like outside the context
adata_bcr.obs["IGHV_CLL_like"] = np.where(v_heavy.isin(cll_genes), v_heavy, "other")

print(adata_bcr.obs["IGHV_CLL_like"].value_counts())

In [ ]:
ir.pl.group_abundance(
    adata_bcr,
    groupby="disease_stage",
    target_col="IGHV_CLL_like",
    normalize=True,
)

In [ ]:
print("IGHV_CLL_like in obs:", "IGHV_CLL_like" in adata_bcr.obs.columns)
print(adata_bcr.obs[["disease_stage", "IGHV_CLL_like"]].head())

In [ ]:
import numpy as np
import scirpy as ir

cll_genes = ["IGHV1-69", "IGHV4-34", "IGHV3-30", "IGHV3-21", "IGHV1-2"]

with ir.get.airr_context(adata_bcr, "v_call"):
    v = adata_bcr.obs["VDJ_1_v_call"].astype("string")
    adata_bcr.obs["IGHV_CLL_like"] = np.where(v.isin(cll_genes), v, "other")

ir.pl.group_abundance(
    adata_bcr,
    groupby="disease_stage",
    target_col="IGHV_CLL_like",
    normalize=True,
)

In [ ]:
import scirpy as ir

ir.pl.spectratype(
    adata_bcr,
    chain="VDJ_1",          # heavy chain
    color="disease_stage",  # required in your version
    viztype="bar",
    normalize=True,
    fig_kws={"figsize": (6, 4), "dpi": 120},
)

In [ ]:
import numpy as np
import pandas as pd

clone_col = "clone_id"  # or "airr:clone_id" if that's your actual column name

# Per-clone sizes
clone_sizes = adata_bcr.obs[clone_col].value_counts()
clone_sizes.name = "clone_size"

# Map clone size to each cell
adata_bcr.obs["clone_size"] = adata_bcr.obs[clone_col].map(clone_sizes)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,4))
sns.boxplot(
    data=adata_bcr.obs,
    x="disease_stage",
    y=np.log10(adata_bcr.obs["clone_size"]),
)
plt.ylabel("log10(clone size)")
plt.title("BCR clone size distribution by disease stage")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.violinplot(
    data=adata_bcr.obs,
    x="disease_stage",
    y=np.log10(adata_bcr.obs["clone_size"]),
    cut=0,
    scale="width",
)
plt.ylabel("log10(clone size)")
plt.title("BCR clone size distribution by disease stage")
plt.tight_layout()
plt.show()

In [ ]:
# per-patient fraction of cells in expanded clones (size >=2)
adata_bcr.obs["expanded"] = adata_bcr.obs["clonal_expansion"] != "<= 1"

clonal_df = (
    adata_bcr.obs
    .groupby(["patient_id", "disease_stage"])["expanded"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(6,4))
sns.barplot(
    data=clonal_df,
    x="patient_id",
    y="expanded",
    hue="disease_stage",
)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Fraction of BCR cells in expanded clones")
plt.title("Per-patient clonal expansion")
plt.tight_layout()
plt.show()